In [ ]:
import os
import json
import math
import copy
import random
import warnings
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import torch 
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms, models
from torchvision.models import ResNet18_Weights
from torchvision.models.detection import (
    FasterRCNN_ResNet50_FPN_Weights,
    fasterrcnn_resnet50_fpn,
)
from torchvision.ops import box_iou
from PIL import Image

warnings.filterwarnings("ignore")

SEED = 42
BATCH_SIZE = 256
NUM_WORKERS = 0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Часть A
CLS_DATASET_NAME = "CIFAR100"
DOWNLOAD_CLS = True     
CLS_EPOCHS_C1 = 5
CLS_EPOCHS_C2 = 5
CLS_EPOCHS_C3 = 4
CLS_EPOCHS_C4 = 4
CLS_LR_CNN = 1e-3
CLS_LR_HEAD = 1e-3
CLS_LR_FT = 3e-4

# Часть B
DET_DATASET_NAME = "Pascal VOC"
VOC_YEAR = "2012"
DET_SCORE_THR_V1 = 0.3
DET_SCORE_THR_V2 = 0.7
DET_MAX_SAMPLES = 10
DOWNLOAD_DET_DATA = False    
DOWNLOAD_DET_WEIGHTS = True  


ROOT = Path(".")
ART = ROOT / "artifacts"
FIG = ART / "figures"
DATA = ROOT / "data"

ART.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)
DATA.mkdir(parents=True, exist_ok=True)


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

print(f"Устройство: {DEVICE}")
print(f"Seed: {SEED}")
print(f"Рабочая папка: {ROOT.resolve()}")


def train_one_epoch_classification(model, loader, criterion, optimizer, device):
    model.train()
    loss_sum, acc_sum, n = 0.0, 0.0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        bs = x.size(0)
        loss_sum += loss.item() * bs
        acc_sum += (logits.argmax(dim=1) == y).float().sum().item()
        n += bs

    return loss_sum / max(n, 1), acc_sum / max(n, 1)

@torch.no_grad()
def evaluate_classification(model, loader, criterion, device):
    model.eval()
    loss_sum, acc_sum, n = 0.0, 0.0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)

        bs = x.size(0)
        loss_sum += loss.item() * bs
        acc_sum += (logits.argmax(dim=1) == y).float().sum().item()
        n += bs

    return loss_sum / max(n, 1), acc_sum / max(n, 1)

class TransformSubset(torch.utils.data.Dataset):
    def __init__(self, subset, transform=None, target_transform=None):
        self.subset = subset
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        x, y = self.subset[idx]
        if self.transform is not None:
            x = self.transform(x)
        if self.target_transform is not None:
            y = self.target_transform(y)
        return x, y

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=100):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

def build_resnet18_head_only(num_classes):
    weights = ResNet18_Weights.DEFAULT
    model = models.resnet18(weights=weights)
    for p in model.parameters():
        p.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

def build_resnet18_finetune(num_classes):
    weights = ResNet18_Weights.DEFAULT
    model = models.resnet18(weights=weights)
    for p in model.parameters():
        p.requires_grad = False
    for p in model.layer4.parameters():
        p.requires_grad = True
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

def plot_curves(history, title, save_path):
    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(10, 4))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history["train_loss"], label="train_loss")
    plt.plot(epochs, history["val_loss"], label="val_loss")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title(f"{title} | loss")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history["train_acc"], label="train_acc")
    plt.plot(epochs, history["val_acc"], label="val_acc")
    plt.xlabel("epoch")
    plt.ylabel("accuracy")
    plt.title(f"{title} | accuracy")
    plt.legend()

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()

def save_aug_preview(dataset_raw, aug_transform, path, n=6):
    plt.figure(figsize=(12, 4))
    for i in range(n):
        img, _ = dataset_raw[i]
        aug_img = aug_transform(img)
        aug_img = aug_img.permute(1, 2, 0).cpu().numpy()
        aug_img = np.clip(aug_img, 0, 1)

        plt.subplot(2, math.ceil(n / 2), i + 1)
        plt.imshow(aug_img)
        plt.axis("off")
        plt.title(f"aug {i+1}")

    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()

def run_classification_experiment(
    experiment_id,
    model,
    train_loader,
    val_loader,
    test_loader,
    optimizer,
    criterion,
    epochs,
    dataset_name,
    model_summary,
    lr,
    notes=""
):
    model = model.to(DEVICE)
    best_state = None
    best_val_acc = -1.0
    best_val_loss = None
    best_epoch = 0

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_one_epoch_classification(model, train_loader, criterion, optimizer, DEVICE)
        va_loss, va_acc = evaluate_classification(model, val_loader, criterion, DEVICE)

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)

        print(f"{experiment_id} | эпоха {epoch:02d}/{epochs} | train_loss={tr_loss:.4f} | train_acc={tr_acc:.4f} | val_loss={va_loss:.4f} | val_acc={va_acc:.4f}")

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            best_val_loss = va_loss
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    _, test_acc = evaluate_classification(model, test_loader, criterion, DEVICE)

    result = {
        "experiment_id": experiment_id,
        "task": "classification",
        "dataset": dataset_name,
        "seed": SEED,
        "model_summary": model_summary,
        "optimizer": optimizer.__class__.__name__,
        "lr": lr,
        "epochs_trained": epochs,
        "best_val_accuracy": float(best_val_acc),
        "test_accuracy": float(test_acc),
        "precision": np.nan,
        "recall": np.nan,
        "mean_iou": np.nan,
        "notes": notes + f"; best_epoch={best_epoch}; best_val_loss={best_val_loss:.4f}"
    }
    return model, history, result

def make_loader(ds, batch_size=BATCH_SIZE, shuffle=False):
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available()
    )

print("\n" + "=" * 80)
print("ЧАСТЬ A | CIFAR100 | C1-C4")
print("=" * 80)

DATA = Path("data")

train_raw = datasets.CIFAR100(
    root=DATA,
    train=True,
    download=False
)

test_raw = datasets.CIFAR100(
    root=DATA,
    train=False,
    download=False
)

num_classes = 100
class_names = train_raw.classes

g = torch.Generator().manual_seed(SEED)
train_size = int(0.8 * len(train_raw))
val_size = len(train_raw) - train_size
train_subset_raw, val_subset_raw = random_split(train_raw, [train_size, val_size], generator=g)

cnn_base_tf = transforms.Compose([
    transforms.ToTensor(),
])

cnn_aug_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
])

resnet_weights = ResNet18_Weights.DEFAULT
resnet_tf = resnet_weights.transforms()

train_c1 = TransformSubset(train_subset_raw, transform=cnn_base_tf)
val_c1 = TransformSubset(val_subset_raw, transform=cnn_base_tf)
test_c1 = TransformSubset(test_raw, transform=cnn_base_tf)

train_c2 = TransformSubset(train_subset_raw, transform=cnn_aug_tf)
val_c2 = TransformSubset(val_subset_raw, transform=cnn_base_tf)
test_c2 = TransformSubset(test_raw, transform=cnn_base_tf)

train_c3 = TransformSubset(train_subset_raw, transform=resnet_tf)
val_c3 = TransformSubset(val_subset_raw, transform=resnet_tf)
test_c3 = TransformSubset(test_raw, transform=resnet_tf)

train_c4 = TransformSubset(train_subset_raw, transform=resnet_tf)
val_c4 = TransformSubset(val_subset_raw, transform=resnet_tf)
test_c4 = TransformSubset(test_raw, transform=resnet_tf)

sanity_loader = DataLoader(train_c1, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
x_batch, y_batch = next(iter(sanity_loader))

print(f"Датасет: {CLS_DATASET_NAME}")
print(f"Размер train: {len(train_c1)}")
print(f"Размер val:   {len(val_c1)}")
print(f"Размер test:  {len(test_c1)}")
print(f"Форма x: {x_batch.shape}")
print(f"Форма y: {y_batch.shape}")
print(f"Минимум x: {x_batch.min().item():.4f} | Максимум x: {x_batch.max().item():.4f}")

save_aug_preview(train_raw, cnn_aug_tf, FIG / "augmentations_preview.png", n=6)

# C1
train_loader_c1 = make_loader(train_c1, shuffle=True)
val_loader_c1 = make_loader(val_c1)
test_loader_c1 = make_loader(test_c1)

model_c1 = SimpleCNN(num_classes=num_classes)
optim_c1 = torch.optim.Adam(model_c1.parameters(), lr=CLS_LR_CNN)
criterion = nn.CrossEntropyLoss()

model_c1, hist_c1, res_c1 = run_classification_experiment(
    experiment_id="C1",
    model=model_c1,
    train_loader=train_loader_c1,
    val_loader=val_loader_c1,
    test_loader=test_loader_c1,
    optimizer=optim_c1,
    criterion=criterion,
    epochs=CLS_EPOCHS_C1,
    dataset_name=CLS_DATASET_NAME,
    model_summary="simple-cnn-base",
    lr=CLS_LR_CNN,
    notes="base CNN without augmentations",
)

# C2
train_loader_c2 = make_loader(train_c2, shuffle=True)
val_loader_c2 = make_loader(val_c2)
test_loader_c2 = make_loader(test_c2)

model_c2 = SimpleCNN(num_classes=num_classes)
optim_c2 = torch.optim.Adam(model_c2.parameters(), lr=CLS_LR_CNN)

model_c2, hist_c2, res_c2 = run_classification_experiment(
    experiment_id="C2",
    model=model_c2,
    train_loader=train_loader_c2,
    val_loader=val_loader_c2,
    test_loader=test_loader_c2,
    optimizer=optim_c2,
    criterion=criterion,
    epochs=CLS_EPOCHS_C2,
    dataset_name=CLS_DATASET_NAME,
    model_summary="simple-cnn-aug",
    lr=CLS_LR_CNN,
    notes="same CNN with augmentations",
)

# C3
train_loader_c3 = make_loader(train_c3, shuffle=True)
val_loader_c3 = make_loader(val_c3)
test_loader_c3 = make_loader(test_c3)

model_c3 = build_resnet18_head_only(num_classes=num_classes)
optim_c3 = torch.optim.Adam(filter(lambda p: p.requires_grad, model_c3.parameters()), lr=CLS_LR_HEAD)

model_c3, hist_c3, res_c3 = run_classification_experiment(
    experiment_id="C3",
    model=model_c3,
    train_loader=train_loader_c3,
    val_loader=val_loader_c3,
    test_loader=test_loader_c3,
    optimizer=optim_c3,
    criterion=criterion,
    epochs=CLS_EPOCHS_C3,
    dataset_name=CLS_DATASET_NAME,
    model_summary="resnet18-head-only",
    lr=CLS_LR_HEAD,
    notes="pretrained ResNet18, frozen backbone, train fc only",
)

# C4
train_loader_c4 = make_loader(train_c4, shuffle=True)
val_loader_c4 = make_loader(val_c4)
test_loader_c4 = make_loader(test_c4)

model_c4 = build_resnet18_finetune(num_classes=num_classes)
optim_c4 = torch.optim.Adam(filter(lambda p: p.requires_grad, model_c4.parameters()), lr=CLS_LR_FT)

model_c4, hist_c4, res_c4 = run_classification_experiment(
    experiment_id="C4",
    model=model_c4,
    train_loader=train_loader_c4,
    val_loader=val_loader_c4,
    test_loader=test_loader_c4,
    optimizer=optim_c4,
    criterion=criterion,
    epochs=CLS_EPOCHS_C4,
    dataset_name=CLS_DATASET_NAME,
    model_summary="resnet18-finetune-layer4-fc",
    lr=CLS_LR_FT,
    notes="pretrained ResNet18, fine-tune layer4 + fc",
)

class_results = [res_c1, res_c2, res_c3, res_c4]
histories = {"C1": hist_c1, "C2": hist_c2, "C3": hist_c3, "C4": hist_c4}
models_map = {"C1": model_c1, "C2": model_c2, "C3": model_c3, "C4": model_c4}

best_cls = max(class_results, key=lambda r: r["best_val_accuracy"])
best_id = best_cls["experiment_id"]
best_model = models_map[best_id]
best_history = histories[best_id]

print(f"\nЛучшая классификационная модель: {best_id} | val_acc={best_cls['best_val_accuracy']:.4f} | test_acc={best_cls['test_accuracy']:.4f}")

torch.save(best_model.state_dict(), ART / "best_classifier.pt")

best_config = {
    "dataset": CLS_DATASET_NAME,
    "experiment_id": best_id,
    "model_name": best_cls["model_summary"],
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "optimizer": best_cls["optimizer"],
    "lr": best_cls["lr"],
    "epochs_trained": best_cls["epochs_trained"],
    "best_val_accuracy": best_cls["best_val_accuracy"],
    "test_accuracy": best_cls["test_accuracy"],
    "base_transforms": "ToTensor()",
    "augmentation_transforms": "RandomHorizontalFlip + RandomCrop(32,padding=4) + ColorJitter + ToTensor()",
    "resnet_transforms": str(resnet_weights.transforms()),
}

with open(ART / "best_classifier_config.json", "w", encoding="utf-8") as f:
    json.dump(best_config, f, ensure_ascii=False, indent=2)

plot_curves(best_history, f"Лучший прогон {best_id}", FIG / "classification_curves_best.png")

plt.figure(figsize=(8, 4))
ids = [r["experiment_id"] for r in class_results]
vals = [r["best_val_accuracy"] for r in class_results]
plt.bar(ids, vals)
for i, v in enumerate(vals):
    plt.text(i, v + 0.005, f"{v:.3f}", ha="center")
plt.ylim(0, min(1.0, max(vals) + 0.12))
plt.xlabel("experiment_id")
plt.ylabel("best_val_accuracy")
plt.title("Сравнение C1-C4")
plt.tight_layout()
plt.savefig(FIG / "classification_compare.png", dpi=150, bbox_inches="tight")
plt.close()

print("\n" + "=" * 80)
print("ЧАСТЬ B | Pascal VOC | detection | V1-V2")
print("=" * 80)

VOC_ROOT = DATA / "VOCdevkit" / f"VOC{VOC_YEAR}"

if not VOC_ROOT.exists():
    raise FileNotFoundError(
        f"Не найдена папка {VOC_ROOT}. "
        f"По твоей структуре Pascal VOC должен лежать в data/VOCdevkit/VOC{VOC_YEAR}."
    )

VOC_CLASSES = [
    "__background__", "aeroplane", "bicycle", "bird", "boat", "bottle",
    "bus", "car", "cat", "chair", "cow", "diningtable", "dog", "horse",
    "motorbike", "person", "pottedplant", "sheep", "sofa", "train", "tvmonitor"
]
VOC_CLASS_TO_IDX = {name: idx for idx, name in enumerate(VOC_CLASSES)}

class VOCDatasetForDetection(torch.utils.data.Dataset):
    def __init__(self, root_dir, split_file, image_transform=None):
        self.root_dir = Path(root_dir)
        self.image_dir = self.root_dir / "JPEGImages"
        self.ann_dir = self.root_dir / "Annotations"
        self.image_transform = image_transform

        with open(split_file, "r", encoding="utf-8") as f:
            self.ids = [line.strip() for line in f.readlines() if line.strip()]

    def __len__(self):
        return len(self.ids)

    def _parse_annotation(self, ann_path):
        tree = ET.parse(ann_path)
        root = tree.getroot()

        boxes = []
        labels = []

        for obj in root.findall("object"):
            name = obj.find("name").text
            if name not in VOC_CLASS_TO_IDX:
                continue

            bnd = obj.find("bndbox")
            xmin = float(bnd.find("xmin").text)
            ymin = float(bnd.find("ymin").text)
            xmax = float(bnd.find("xmax").text)
            ymax = float(bnd.find("ymax").text)

            if xmax > xmin and ymax > ymin:
                boxes.append([xmin, ymin, xmax, ymax])
                labels.append(VOC_CLASS_TO_IDX[name])

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)

        return boxes, labels

    def __getitem__(self, idx):
        image_id = self.ids[idx]
        img_path = self.image_dir / f"{image_id}.jpg"
        ann_path = self.ann_dir / f"{image_id}.xml"

        img = Image.open(img_path).convert("RGB")
        boxes, labels = self._parse_annotation(ann_path)

        raw_img = transforms.ToTensor()(img)

        if self.image_transform is not None:
            img = self.image_transform(img)
        else:
            img = transforms.ToTensor()(img)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
        }
        return img, target, raw_img, image_id

def det_collate_fn(batch):
    imgs = [item[0] for item in batch]
    targets = [item[1] for item in batch]
    raw_imgs = [item[2] for item in batch]
    image_ids = [item[3] for item in batch]
    return imgs, targets, raw_imgs, image_ids

det_weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT if DOWNLOAD_DET_WEIGHTS else None
det_model = fasterrcnn_resnet50_fpn(weights=det_weights).to(DEVICE)
det_model.eval()

det_transform = FasterRCNN_ResNet50_FPN_Weights.DEFAULT.transforms() if det_weights is not None else transforms.ToTensor()

split_path = VOC_ROOT / "ImageSets" / "Main" / "val.txt"
if not split_path.exists():
    split_path = VOC_ROOT / "ImageSets" / "Main" / "test.txt"

det_ds = VOCDatasetForDetection(
    root_dir=VOC_ROOT,
    split_file=split_path,
    image_transform=det_transform,
)

if DET_MAX_SAMPLES is not None and DET_MAX_SAMPLES < len(det_ds):
    det_ds = Subset(det_ds, list(range(DET_MAX_SAMPLES)))

det_loader = DataLoader(
    det_ds,
    batch_size=4,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=det_collate_fn,
    pin_memory=torch.cuda.is_available(),
)

print(f"Detection dataset: {DET_DATASET_NAME}")
print(f"Количество примеров для оценки: {len(det_ds)}")

def match_predictions_to_gt(pred_boxes, pred_labels, gt_boxes, gt_labels, iou_thr=0.5):
    if len(pred_boxes) == 0 or len(gt_boxes) == 0:
        return 0, 0, 0, []

    ious = box_iou(pred_boxes, gt_boxes)
    matches = []
    used_gt = set()

    pred_order = torch.arange(len(pred_boxes))
    for i in pred_order:
        label_mask = (gt_labels == pred_labels[i])
        if label_mask.sum() == 0:
            continue

        candidate_ious = ious[i].clone()
        candidate_ious[~label_mask] = -1.0
        best_iou, best_gt = torch.max(candidate_ious, dim=0)

        if best_iou.item() >= iou_thr and int(best_gt.item()) not in used_gt:
            used_gt.add(int(best_gt.item()))
            matches.append((int(i.item()), int(best_gt.item()), float(best_iou.item())))

    tp = len(matches)
    fp = len(pred_boxes) - tp
    fn = len(gt_boxes) - tp
    return tp, fp, fn, matches

@torch.no_grad()
def evaluate_detection(model, loader, score_thr, device):
    total_tp, total_fp, total_fn = 0, 0, 0
    matched_ious = []
    examples = []

    for imgs, targets, raw_imgs, image_ids in loader:
        imgs_dev = [img.to(device) for img in imgs]
        outputs = model(imgs_dev)

        for img, target, raw_img, image_id, output in zip(imgs, targets, raw_imgs, image_ids, outputs):
            pred_boxes = output["boxes"].detach().cpu()
            pred_labels = output["labels"].detach().cpu()
            pred_scores = output["scores"].detach().cpu()

            keep = pred_scores >= score_thr
            pred_boxes = pred_boxes[keep]
            pred_labels = pred_labels[keep]

            gt_boxes = target["boxes"].detach().cpu()
            gt_labels = target["labels"].detach().cpu()

            tp, fp, fn, matches = match_predictions_to_gt(
                pred_boxes, pred_labels, gt_boxes, gt_labels, iou_thr=0.5
            )

            total_tp += tp
            total_fp += fp
            total_fn += fn
            matched_ious.extend([m[2] for m in matches])

            if len(examples) < 4:
                examples.append({
                    "image": raw_img.permute(1, 2, 0).cpu().numpy(),
                    "gt_boxes": gt_boxes.numpy() if len(gt_boxes) else np.zeros((0, 4)),
                    "pred_boxes": pred_boxes.numpy() if len(pred_boxes) else np.zeros((0, 4)),
                    "image_id": image_id
                })

    precision = total_tp / max(total_tp + total_fp, 1)
    recall = total_tp / max(total_tp + total_fn, 1)
    mean_iou = float(np.mean(matched_ious)) if len(matched_ious) > 0 else 0.0

    return {
        "precision": float(precision),
        "recall": float(recall),
        "mean_iou": float(mean_iou),
        "examples": examples,
    }

det_v1 = evaluate_detection(det_model, det_loader, DET_SCORE_THR_V1, DEVICE)
det_v2 = evaluate_detection(det_model, det_loader, DET_SCORE_THR_V2, DEVICE)

print(f"V1 | precision={det_v1['precision']:.4f} | recall={det_v1['recall']:.4f} | mean_iou={det_v1['mean_iou']:.4f}")
print(f"V2 | precision={det_v2['precision']:.4f} | recall={det_v2['recall']:.4f} | mean_iou={det_v2['mean_iou']:.4f}")

# detection_examples.png
plt.figure(figsize=(12, 10))
for i, ex in enumerate(det_v1["examples"]):
    img = np.clip(ex["image"], 0, 1)
    ax = plt.subplot(2, 2, i + 1)
    ax.imshow(img)
    ax.set_title(f"pred V1 | {ex['image_id']}")
    ax.axis("off")

    for box in ex["gt_boxes"]:
        x1, y1, x2, y2 = box
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, linewidth=2)
        ax.add_patch(rect)

    for box in ex["pred_boxes"]:
        x1, y1, x2, y2 = box
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, linestyle="--", linewidth=2)
        ax.add_patch(rect)

plt.tight_layout()
plt.savefig(FIG / "detection_examples.png", dpi=150, bbox_inches="tight")
plt.close()

# detection_metrics.png
plt.figure(figsize=(8, 4))
metric_names = ["precision", "recall", "mean_iou"]
v1_vals = [det_v1[m] for m in metric_names]
v2_vals = [det_v2[m] for m in metric_names]

x = np.arange(len(metric_names))
w = 0.35

plt.bar(x - w / 2, v1_vals, width=w, label="V1")
plt.bar(x + w / 2, v2_vals, width=w, label="V2")

for i, v in enumerate(v1_vals):
    plt.text(i - w / 2, v + 0.01, f"{v:.3f}", ha="center", fontsize=9)
for i, v in enumerate(v2_vals):
    plt.text(i + w / 2, v + 0.01, f"{v:.3f}", ha="center", fontsize=9)

plt.xticks(x, metric_names)
plt.ylim(0, 1.05)
plt.title("Detection metrics: V1 vs V2")
plt.legend()
plt.tight_layout()
plt.savefig(FIG / "detection_metrics.png", dpi=150, bbox_inches="tight")
plt.close()

res_v1 = {
    "experiment_id": "V1",
    "task": "detection",
    "dataset": DET_DATASET_NAME,
    "seed": SEED,
    "model_summary": "fasterrcnn_resnet50_fpn_pretrained",
    "optimizer": "",
    "lr": np.nan,
    "epochs_trained": 0,
    "best_val_accuracy": np.nan,
    "test_accuracy": np.nan,
    "precision": det_v1["precision"],
    "recall": det_v1["recall"],
    "mean_iou": det_v1["mean_iou"],
    "notes": f"score_threshold={DET_SCORE_THR_V1}; IoU match threshold=0.5"
}

res_v2 = {
    "experiment_id": "V2",
    "task": "detection",
    "dataset": DET_DATASET_NAME,
    "seed": SEED,
    "model_summary": "fasterrcnn_resnet50_fpn_pretrained",
    "optimizer": "",
    "lr": np.nan,
    "epochs_trained": 0,
    "best_val_accuracy": np.nan,
    "test_accuracy": np.nan,
    "precision": det_v2["precision"],
    "recall": det_v2["recall"],
    "mean_iou": det_v2["mean_iou"],
    "notes": f"score_threshold={DET_SCORE_THR_V2}; IoU match threshold=0.5"
}

runs_df = pd.DataFrame(class_results + [res_v1, res_v2])
runs_df.to_csv(ART / "runs.csv", index=False, encoding="utf-8")

print("\nСохранён runs.csv")
print(runs_df)



print("\n" + "=" * 80)
print("Создано:")
print(f"- {ART / 'runs.csv'}")
print(f"- {ART / 'best_classifier.pt'}")
print(f"- {ART / 'best_classifier_config.json'}")
print(f"- {FIG / 'classification_curves_best.png'}")
print(f"- {FIG / 'classification_compare.png'}")
print(f"- {FIG / 'augmentations_preview.png'}")
print(f"- {FIG / 'detection_examples.png'}")
print(f"- {FIG / 'detection_metrics.png'}")

print("\nИтоги:")
print(runs_df[[
    "experiment_id", "task", "dataset", "best_val_accuracy", "test_accuracy", "precision", "recall", "mean_iou"
]].to_string(index=False))

Устройство: cpu
Seed: 42
Рабочая папка: C:\Users\kiril\OneDrive\Desktop\ИИИ\REP\III_2025-26\homeworks\HW10-11

ЧАСТЬ A | CIFAR100 | C1-C4
Датасет: CIFAR100
Размер train: 40000
Размер val:   10000
Размер test:  10000
Форма x: torch.Size([256, 3, 32, 32])
Форма y: torch.Size([256])
Минимум x: 0.0000 | Максимум x: 1.0000
C1 | эпоха 01/5 | train_loss=4.3706 | train_acc=0.0290 | val_loss=4.1277 | val_acc=0.0565
C1 | эпоха 02/5 | train_loss=4.0579 | train_acc=0.0640 | val_loss=3.9194 | val_acc=0.0860
C1 | эпоха 03/5 | train_loss=3.8677 | train_acc=0.0904 | val_loss=3.7678 | val_acc=0.1135
C1 | эпоха 04/5 | train_loss=3.7394 | train_acc=0.1056 | val_loss=3.6311 | val_acc=0.1321
C1 | эпоха 05/5 | train_loss=3.6324 | train_acc=0.1221 | val_loss=3.5498 | val_acc=0.1442
C2 | эпоха 01/5 | train_loss=4.4179 | train_acc=0.0242 | val_loss=4.2057 | val_acc=0.0486
C2 | эпоха 02/5 | train_loss=4.1331 | train_acc=0.0543 | val_loss=3.9460 | val_acc=0.0733
C2 | эпоха 03/5 | train_loss=3.9510 | train_acc=0.

100.0%
